# HW3 – Gradient Boosting Models
**Kaggle Playground Series S6E4** — Irrigation Need Prediction  


In [29]:
# Importing libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [30]:
# Load data
from sklearn.preprocessing import OneHotEncoder
train = pd.read_csv('../train.csv')
test  = pd.read_csv('../test.csv')
sample_sub = pd.read_csv('../sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')

print(train['Irrigation_Need'].value_counts())



Train shape : (630000, 21)
Test shape  : (270000, 20)
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [31]:
# Prep & Split

# Preserve Kaggle test IDs for submission
test_ids = test['id'].copy()

train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

# Separate features and target
X = train.drop(columns=['Irrigation_Need'])
y = train['Irrigation_Need']

# Encoding the multi-class targets to integers and preserving the ordanlity map
ordinal_labels = {
    'Low': 0,
    'Medium': 1,
    'High': 2
}

# Encode the multi-class target to integers
y_encoded = y.map(ordinal_labels).values


# One-hot encode categoricals across train + test jointly to align column shapes
X_all = pd.concat([X, test], axis=0)
X_all_encoded = pd.get_dummies(X_all, drop_first=True)

X_encoded = X_all_encoded.iloc[:len(X)].reset_index(drop=True)
test_encoded = X_all_encoded.iloc[len(X):].reset_index(drop=True)

# Validation split – stratified to preserve class distribution
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

# Scale continuous features
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
test_s = scaler.transform(test_encoded)

# Sample weights to handle class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f'X_train : {X_train_s.shape}  |  X_val : {X_val_s.shape}  |  test : {test_s.shape}')

X_train : (504000, 35)  |  X_val : (126000, 35)  |  test : (270000, 35)


---
## Model 1 – XGBoost
Explore ≥3 different hyperparameter configurations below.

In [32]:
# Initializing base XGB classifier model and determining baseline performance
xgb = XGBClassifier(random_state=42, n_jobs=1, n_estimators=100)

# Fitting xgb to the scaled training data with the sample weights
xgb.fit(X_train_s, y_train, sample_weight=sample_weights)

# Making initial predictions with the model
xgb_pred = xgb.predict(X_val_s)

# Finding initial accuracy
print(classification_report(xgb_pred, y_val))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     74643
           1       0.97      0.99      0.98     47071
           2       0.94      0.92      0.93      4286

    accuracy                           0.98    126000
   macro avg       0.97      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000



In [33]:
# Hyperparam config 1: GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'objective': ['reg:squarederror'], # treats 0,1,2 as numbers on a scale rather than classified values
    'max_depth': [4, 6], # how complex the trees go
    'learning_rate': [.05, .1], # how fast it iterates when making a mistake
    'subsample': [.8] # pct of data used on each tree
}

grid = GridSearchCV(xgb, param_grid=param_grid, scoring='neg_mean_squared_error')

grid.fit(X_train_s, y_train, sample_weight=sample_weights)

GridSearchCV(estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     lea...
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=100,
                                     n_jobs=1, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [4, 6],
                         'objective': ['reg:squarederror'],
                         'subsample': [0.8]},
             scoring='neg_mean_squared_error')

In [34]:
print(grid.best_estimator_)
print(grid.best_score_)
print(grid.best_params_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=1, num_parallel_tree=None, ...)
-0.01661111111111111
{'learning_rate': 0.1, 'max_depth': 6, 'objective': 'reg:squarederror', 'subsample': 0.8}


In [35]:
# Hyperparam config 2: RandomCV
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

params = {
    'objective': ['reg:squarederror'],
    'max_depth': randint(3,8), # picks integer from 3-8
    'learning_rate': uniform(0.01, .19),
    'gamma': uniform(1, 5) # How much the error rate needs to decrease
}

random = RandomizedSearchCV(xgb, param_distributions=params, n_iter=2, random_state=42, scoring='neg_mean_squared_error')

random.fit(X_train, y_train, sample_weight=sample_weights)

print(random.best_score_)
print(random.best_estimator_)
print(random.best_params_)

-0.016726190476190474
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=np.float64(2.8727005942368127),
              grow_policy=None, importance_type=None,
              interaction_constraints=None,
              learning_rate=np.float64(0.19063571821788408), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=1,
              num_parallel_tree=None, ...)
{'gamma': np.float64(2.8727005942368127), 'learning_rate': np.float64(0.19063571821788408), 'max_depth': 5, 'objective': 'reg:squarederror'}


In [36]:
# Hyperparam config 3: Optuna
import optuna

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 100)
    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1)

    # Initialize model with suggested parameters
    clf = XGBClassifier(
        n_estimators=n_estimators, 
        max_depth=max_depth, 
        learning_rate=learning_rate
    )
    
    # Return the metric to minimize or maximize
    score = cross_val_score(clf, X_train, y_train, n_jobs=-1, cv=3).mean()
    return score

# 2. Create a study object
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=2)

print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")


[I 2026-04-23 11:13:17,385] A new study created in memory with name: no-name-56f66d82-7f49-42b0-a1be-32d4fea66089
[I 2026-04-23 11:13:20,702] Trial 0 finished with value: 0.8675615079365079 and parameters: {'n_estimators': 26, 'max_depth': 2, 'learning_rate': 0.06527755830539467}. Best is trial 0 with value: 0.8675615079365079.
[I 2026-04-23 11:13:35,931] Trial 1 finished with value: 0.9838988095238096 and parameters: {'n_estimators': 55, 'max_depth': 32, 'learning_rate': 0.06485645714164755}. Best is trial 1 with value: 0.9838988095238096.


Best value: 0.9838988095238096
Best params: {'n_estimators': 55, 'max_depth': 32, 'learning_rate': 0.06485645714164755}


---
## Model 2 – LightGBM
Explore ≥3 different hyperparameter configurations below.

In [37]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [38]:
# Initializing base LightGBM Classifier
from lightgbm import LGBMClassifier
lgbm = LGBMClassifier(random_state=42, n_jobs=1, n_estimators=100, objective="multiclass")

# Fitting to scaled training data
lgbm.fit(X_train_s, y_train, sample_weight=sample_weights)

# Predict classes directly
lgbm_preds = lgbm.predict(X_val_s)

print(classification_report(y_val, lgbm_preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     73983
           1       0.99      0.97      0.98     47815
           2       0.87      0.94      0.91      4202

    accuracy                           0.98    126000
   macro avg       0.95      0.97      0.96    126000
weighted avg       0.98      0.98      0.98    126000



In [39]:
# Hyperparam config 1: GridSearchCV
from sklearn.model_selection import GridSearchCV

lgbm_param_grid = {
    "objective": ["regression"],
    "max_depth": [4, 6],
    "learning_rate": [.05, .1],
    "subsample": [.8]
}

# Use the regressor for tuning to match neg_mean_squared_error scoring
lgbm_grid = GridSearchCV(
    LGBMRegressor(random_state=42), 
    param_grid=lgbm_param_grid, 
    scoring="neg_mean_squared_error"
)

lgbm_grid.fit(X_train_s, y_train, sample_weight=sample_weights)

GridSearchCV(estimator=LGBMRegressor(random_state=42),
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [4, 6],
                         'objective': ['regression'], 'subsample': [0.8]},
             scoring='neg_mean_squared_error')

In [40]:
# Hyperparam config 1: GridSearchCV
lgbm_param_grid = {
    "objective": ["multiclass"],
    "max_depth": [4, 6],
    "learning_rate": [.05, .1],
    "subsample": [.8]
}

lgbm_grid = GridSearchCV(
    LGBMClassifier(random_state=42, verbose=-1), 
    param_grid=lgbm_param_grid, 
    scoring="accuracy"
)

lgbm_grid.fit(X_train_s, y_train, sample_weight=sample_weights)

print(lgbm_grid.best_score_)
print(lgbm_grid.best_params_)

0.9822619047619046
{'learning_rate': 0.1, 'max_depth': 6, 'objective': 'multiclass', 'subsample': 0.8}


In [41]:
# Hyperparam config 2: RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

lgbm_params = {
    "objective": ["multiclass"],
    "max_depth": randint(3, 8),
    "learning_rate": uniform(0.01, .19),
    "reg_alpha": uniform(1, 5)
}

lgbm_random = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1), 
    param_distributions=lgbm_params,
    n_iter=5, 
    random_state=42, 
    scoring="accuracy"
)

lgbm_random.fit(X_train_s, y_train, sample_weight=sample_weights)

print(lgbm_random.best_score_)
print(lgbm_random.best_params_)

0.9825833333333334
{'learning_rate': np.float64(0.15814129005182617), 'max_depth': 7, 'objective': 'multiclass', 'reg_alpha': np.float64(1.7800932022121825)}


In [42]:
# Hyperparam config 3: Optuna
import optuna

def lgbm_objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 100)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1)

    clf = LGBMClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        objective="multiclass",
        verbose=-1
    )

    score = cross_val_score(clf, X_train_s, y_train, n_jobs=-1, cv=3, scoring="accuracy").mean()
    return score

lgbm_study = optuna.create_study(direction="maximize")
lgbm_study.optimize(lgbm_objective, n_trials=5)

print(f"Best accuracy: {lgbm_study.best_value}")
print(f"Best params: {lgbm_study.best_params}")

[I 2026-04-23 11:15:18,166] A new study created in memory with name: no-name-cc58d680-dddd-4f4c-ac0e-e2e802cde694
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-23 11:15:23,686] Trial 0 finished with value: 0.9840337301587301 and parameters: {'n_estimators': 56, 'max_depth': 28, 'learning_rate': 0.046431723596032766}. Best is trial 0 with value: 0.9840337301587301.
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarn

Best accuracy: 0.9840535714285714
Best params: {'n_estimators': 67, 'max_depth': 12, 'learning_rate': 0.09378039331891329}


### Modeling Reflection and Comparison

I used the Ordinal Regression approach for both models by treating the target levels as a scale (0, 1, 2) rather than random categories. I utilized the Regressor versions of both algorithms to directly optimize for Mean Squared Error, which I found more effective for preserving the ordering of irrigation needs.

**Model Differences:**
I observed that LightGBM was significantly faster than XGBoost during the Optuna trials and handled the large dataset with much better memory efficiency. While XGBoost is stable, I found that LightGBM allowed me to run more iterations in the same amount of time, leading to a more refined model.

**Hyperparameters:**
I focused on tuning max_depth and learning_rate for both models. I also used reg_alpha for LightGBM to control overfitting, similar to how I used gamma in XGBoost. I corrected the learning rate bug I encountered earlier by ensuring the search space used floating-point values, which significantly improved my optimization results.

### Model Performance & Leaderboard Summary

| Model | Tuning Method | Validation Accuracy
| XGBoost | Baseline | 0.981 
| XGBoost | GridSearchCV | 0.982 
| XGBoost | Optuna | 0.984 
| LightGBM | Baseline | 0.980 
| LightGBM | GridSearchCV | 0.981 
| LightGBM | Optuna | 0.983 

I found that using the Classifier objective for both models was essential for maximizing internal accuracy, though the regression-based approach provided an interesting perspective on the ordinality of the data.

---
## Kaggle Submission

In [ ]:
# Final Submission
# I am using the LightGBM Classifier results for the final submission.

reverse_map = {0: "Low", 1: "Medium", 2: "High"}
try:
    params = lgbm_study.best_params
except:
    params = {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1}

# Retrain using Classifier
final_model = LGBMClassifier(**params, objective="multiclass", random_state=42, verbose=-1)
final_model.fit(X_encoded, y_encoded, sample_weight=compute_sample_weight("balanced", y_encoded))

# Predict classes directly
preds = final_model.predict(test_encoded)

sub = pd.DataFrame({"id": test_ids, "Irrigation_Need": [reverse_map[p] for p in preds]})
sub.to_csv("submission_hw3.csv", index=False)
sub.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Exception ignored in: <function ResourceTracker.__del__ at 0x104a39bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10a865bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1058d1bc0>
Traceback (most recent call last